# Task 6: 多分类（6 类去向）

**目标**: 预测蛋白质错定位的具体去向（不重定位 / C1同区室 / C2聚集 / C3分泌 / C4核 / C5胞质）。

**注意**: C1=13、C4=29、C2=34，折内只有个位数样本，结果会很抖。**必须打印每类 support**，重点看 C3（分泌，n=121）是否有信号。

In [ ]:
import numpy as np, pandas as pd, warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score,
                              classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

# ===== 加载数据 =====
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")

ID_COLS = ["KEY", "Gene", "reloc_v3"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]

X_full = df_feat[feat_cols].values.astype(np.float32)
y_6class = df_feat["reloc_v3"].values.astype(int)
groups = df_feat["Gene"].values

N_CLASSES = 6
CLASS_NAMES = ["不重定位(C0)", "同区室/C1", "聚集/C2", "分泌途径/C3", "核定位/C4", "细胞质/C5"]

print("6 分类标签分布:")
cls_counts = {}
for i in range(N_CLASSES):
    n = (y_6class == i).sum()
    cls_counts[i] = n
    print(f"  Class {i} ({CLASS_NAMES[i]:20s}): n={n}")
n_total = len(y_6class)
n_pos = (y_6class > 0).sum()
print(f"  总计: n={n_total}, 正例={n_pos}, base_rate={n_pos/n_total:.4f}")

# C1=13 最小, 5折每折仅2-3个, 降到3折
min_cls = min(cls_counts.values())
n_splits = 3 if min_cls < 10 else 5
print(f"\n最小类样本数={min_cls} → 使用 n_splits={n_splits}")
cv_multi = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

## 6a. 多分类 CV 训练

In [ ]:
oof_6 = np.zeros((len(y_6class), N_CLASSES), dtype=np.float32)

for fold, (tr_idx, te_idx) in enumerate(cv_multi.split(X_full, y_6class, groups)):
    X_tr_raw, X_te_raw = X_full[tr_idx], X_full[te_idx]
    y_tr = y_6class[tr_idx]

    imp = SimpleImputer(strategy="median"); scl = StandardScaler()
    X_tr = scl.fit_transform(imp.fit_transform(X_tr_raw)).astype(np.float32)
    X_te = scl.transform(imp.transform(X_te_raw)).astype(np.float32)

    sw = compute_sample_weight("balanced", y_tr)

    model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.5,
                          objective="multi:softprob", eval_metric="mlogloss",
                          num_class=N_CLASSES,
                          random_state=42, n_jobs=-1, tree_method="hist")
    model.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
    oof_6[te_idx] = model.predict_proba(X_te)

    y_te = y_6class[te_idx]
    acc = (oof_6[te_idx].argmax(axis=1) == y_te).mean()
    te_dist = dict(sorted(zip(*np.unique(y_te, return_counts=True))))
    print(f"  Fold {fold}: acc={acc:.3f}, class分布={te_dist}")

## 6b. 结果汇总

In [ ]:
oof_pred = oof_6.argmax(axis=1)
acc = (oof_pred == y_6class).mean()
macro_f1 = f1_score(y_6class, oof_pred, average='macro')
weighted_f1 = f1_score(y_6class, oof_pred, average='weighted')

print(f"\n{'='*60}")
print(f"  多分类结果")
print(f"  Accuracy:    {acc:.4f}")
print(f"  Macro F1:    {macro_f1:.4f}")
print(f"  Weighted F1: {weighted_f1:.4f}")
print(f"{'='*60}")

print(f"\nClassification Report:")
print(classification_report(y_6class, oof_pred, target_names=CLASS_NAMES, digits=4))

print(f"\nConfusion Matrix (行=真实, 列=预测):")
cm = confusion_matrix(y_6class, oof_pred)
header = "           " + " ".join([f"{n:>7s}" for n in [c[:7] for c in CLASS_NAMES]])
print(header)
for i, name in enumerate(CLASS_NAMES):
    row_str = " ".join([f"{v:7d}" for v in cm[i]])
    print(f"  {name:10s} {row_str}  (n={cm[i].sum()})")

print(f"\nPer-Class AUROC (One-vs-Rest):")
for i in range(N_CLASSES):
    y_bin_i = (y_6class == i).astype(int)
    if y_bin_i.sum() > 0:
        auc_i = roc_auc_score(y_bin_i, oof_6[:, i])
        print(f"  Class {i} ({CLASS_NAMES[i]:20s}): "
              f"AUROC={auc_i:.3f}  (support={y_bin_i.sum()})")

print(f"\n⚠ 注意: C1(n=13)、C2(n=34)、C4(n=29) 每折只有个位数样本，"
      f"其 precision/recall 不可过度解读。")
print(f"  重点关注 C3(分泌,n=121) 这类样本较多的是否有信号。")